# 霍尔效应实验数据处理 (Hall Effect)

本 Notebook 用于对霍尔效应实验中直接测得的霍尔电压 $V_H$ 及零磁场电导电压 $V_\sigma$ 进行线性拟合、导电类型判别、物理参数计算（电导率 $\sigma$、霍尔灵敏度 $K_H$、霍尔系数 $R_H$、载流子浓度 $n$ 与迁移率 $\mu$）及完整的不确定度传递分析。

---
### 实验原理与数学公式

#### 1. 霍尔效应基本关系与载流子类型判别
将通有控制工作电流 $I_s$ 的半导体薄片置于垂直磁场 $B$ 中，受洛伦兹力偏转产生横向霍尔电压 $V_H$：
$$V_H = K_H \cdot I_s \cdot B$$
* 若 $V_H > 0$：空穴导电，判定为 **P 型半导体**；
* 若 $V_H < 0$：电子导电，判定为 **N 型半导体**。

#### 2. 磁场计算（螺线管）
励磁电流为 $I_m$，螺线管中心磁场强度满足：
$$B = k_{\mathrm{sol}} \cdot I_m = \left(\mu_0 \frac{N}{L_{\mathrm{sol}}}\right) I_m$$

#### 3. 电导率 $\sigma$ 计算
在零磁场 ($I_m = 0$) 条件下测定不同工作电流 $I_s$ 下沿长度方向的电位差 $V_\sigma$。线性回归斜率即为霍尔片电阻 $R = V_\sigma / I_s$，则材料电导率为：
$$\sigma = \frac{L}{R \cdot b \cdot d}$$
* $L$：电极间距（长度）；$b$：霍尔片宽度；$d$：霍尔片厚度。

#### 4. 霍尔系数 $R_H$、载流子浓度 $n$ 与迁移率 $\mu$
* 霍尔系数：$R_H = |K_H| \cdot d$
* 载流子浓度：$n = \frac{1}{R_H \cdot e} = \frac{1}{|K_H| \cdot e \cdot d}$（$e = 1.602 \times 10^{-19}\,\mathrm{C}$）
* 载流子迁移率：$\mu = \sigma \cdot R_H = \sigma \cdot |K_H| \cdot d$

#### 5. 回归斜率不确定度与相对误差合成
* 线性回归斜率 $k$ 的不确定度：$u(k) = |k| \sqrt{\frac{1/r^2 - 1}{N - 2}}$
* 各物理量相对不确定度传递：
$$u_r(\sigma) \approx \frac{u(R)}{R}, \quad u_r(K_H) \approx \frac{u(k_1)}{k_1}, \quad u_r(n) = u_r(K_H), \quad u_r(\mu) = \sqrt{u_r^2(\sigma) + u_r^2(K_H)}$$

In [ ]:
import math
from python.utils import scientific_round, linear_regression

print("霍尔效应计算与回归分析模块加载完成。")

### 1. 物理常数与霍尔元件几何尺寸

In [ ]:
# 基本物理常量
e_charge = 1.602e-19       # 元电荷 (C)
mu_0 = 4.0 * math.pi * 1e-7  # 真空磁导率 (T·m/A)

# 霍尔元件几何尺寸 (mm -> m)
d_mm = 0.095               # 厚度 d (mm)
b_mm = 0.235               # 宽度 b (mm)
L_mm = 0.270               # 电极间距 L (mm)

d = d_mm * 1e-3
b = b_mm * 1e-3
L = L_mm * 1e-3

# 螺线管磁场比值常数 k_sol = B / Im (T/A)
k_sol = 0.00209

print(f"霍尔元件尺寸: L={L_mm} mm, b={b_mm} mm, d={d_mm} mm")
print(f"螺线管比值: k_sol = {k_sol} T/A")

### 2. 实验测量数据输入
> **包含三组数据**：
> 1. 固定 $I_m$，测量 $V_H$ 随 $I_s$ 的变化；
> 2. 固定 $I_s$，测量 $V_H$ 随 $I_m$ 的变化；
> 3. 零磁场下测量 $V_\sigma$ 随 $I_s$ 的变化。

In [ ]:
# --- 1. VH vs Is 数据 (固定 Im = 600 mA) ---
Im_fixed_mA = 600.0
B_fixed = k_sol * (Im_fixed_mA * 1e-3)  # T
Is_1 = [0.20 * (i + 1) for i in range(10)]  # mA: 0.20 ~ 2.00
# 实测 VH 带符号读数 (mV)
VH_signed_1 = [-1.52, -3.05, -4.58, -6.11, -7.63, -9.16, -10.68, -12.21, -13.74, -15.26]
VH_abs_1 = [abs(x) for x in VH_signed_1]

# --- 2. VH vs Im 数据 (固定 Is = 1.50 mA) ---
Is_fixed_mA = 1.50
Im_2 = [100.0, 150.0, 200.0, 250.0, 300.0, 350.0, 400.0, 450.0, 500.0, 550.0, 600.0]  # mA
VH_signed_2 = [-1.91, -2.87, -3.82, -4.78, -5.73, -6.69, -7.64, -8.60, -9.55, -10.51, -11.46] # mV
VH_abs_2 = [abs(x) for x in VH_signed_2]

# --- 3. Vσ vs Is 数据 (零磁场 Im = 0) ---
Is_3 = list(Is_1) # mA
Vsigma = [45.2, 90.5, 135.8, 181.0, 226.3, 271.5, 316.8, 362.0, 407.3, 452.5]  # mV

print("实验数据载入完成。")

### 3. 线性回归分析与参数计算

In [ ]:
# 1. 回归 1: VH(mV) vs Is(mA)  -> 斜率单位 mV/mA = V/A
k1, b1, r1, uk1 = map(float, linear_regression(Is_1, VH_abs_1))

# 2. 回归 2: VH(mV) vs Im(mA)
k2, b2, r2, uk2 = map(float, linear_regression(Im_2, VH_abs_2))

# 3. 回归 3: Vσ(mV) vs Is(mA)  -> 样品电阻 R (Ω)
R_sample, b3, r3, u_R = map(float, linear_regression(Is_3, Vsigma))

# 导电类型判定
carrier_type = "P 型 (空穴导电)" if sum(VH_signed_1)/len(VH_signed_1) > 0 else "N 型 (电子导电)"

# 电导率 σ = L / (R · b · d)
sigma = L / (R_sample * b * d)
u_r_sigma = abs(u_R / R_sample)
u_sigma = sigma * u_r_sigma
sigma_final, u_sigma_final = scientific_round(sigma, u_sigma)

# 霍尔灵敏度 KH = k1 / B_fixed
KH = k1 / B_fixed
u_r_KH = abs(uk1 / k1)
u_KH = KH * u_r_KH
KH_final, u_KH_final = scientific_round(KH, u_KH)

# 霍尔系数 RH = KH * d
RH = KH * d

# 载流子浓度 n = 1 / (RH · e)
n_carrier = 1.0 / (RH * e_charge)
u_n = n_carrier * u_r_KH
n_final, u_n_final = scientific_round(n_carrier, u_n)

# 迁移率 μ = σ · RH
mu = sigma * RH
u_r_mu = math.sqrt(u_r_sigma**2 + u_r_KH**2)
u_mu = mu * u_r_mu
mu_final, u_mu_final = scientific_round(mu, u_mu)

print("=" * 55)
print("             霍 尔 效 应 实 验 结 果 汇 总            ")
print("=" * 55)
print(f"材料导电类型       : {carrier_type}")
print(f"VH-Is 回归斜率 k1  : {k1:.4f} V/A (r={r1:.4f})")
print(f"VH-Im 回归斜率 k2  : {k2:.4f} V/A (r={r2:.4f})")
print(f"样品内阻 R (斜率k3): {R_sample:.4f} Ω (r={r3:.4f})")
print("-" * 55)
print(f"电导率 σ           : {sigma_final} ± {u_sigma_final} S/m")
print(f"霍尔灵敏度 |KH|    : {KH_final} ± {u_KH_final} V/(A·T)")
print(f"霍尔系数 RH        : {RH:.4e} m³/C")
print(f"载流子浓度 n       : {n_final} ± {u_n_final} m⁻³")
print(f"载流子迁移率 μ     : {mu_final} ± {u_mu_final} m²/(V·s)")
print("-" * 55)
print(f"相对不确定度 u_r(σ) : {u_r_sigma*100:.2f}%")
print(f"相对不确定度 u_r(KH): {u_r_KH*100:.2f}%")
print(f"相对不确定度 u_r(μ) : {u_r_mu*100:.2f}%")
print("=" * 55)